# 4.3 LLM Read-only Debug Agent

这个 notebook 在 4.2 的基础上加入 LLM reasoning。LLM 负责决定下一步调用哪个只读工具，Python 负责执行工具并记录 trajectory。

## 目标

4.2 是 rule-based：Python 固定执行 list/read/search/read。

4.3 是 LLM-assisted：模型根据 task 和工具返回结果决定下一步。

限制仍然很明确：

- 只使用已有 read-only tools。
- 不修改文件。
- 不运行 shell command。
- 不应用 patch。
- 最终只输出诊断和建议。

## 导入依赖并设置目标 workspace

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any, cast

from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import (
    ChatCompletionMessageParam,
    ChatCompletionMessageToolCall,
    ChatCompletionToolParam,
)


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "prototype" else Path.cwd()
AGENT_SRC_ROOT = PROJECT_ROOT / "src"
TARGET_WORKSPACE_ROOT = PROJECT_ROOT / "fixtures" / "workspaces" / "calculator_bug"

os.environ["SWE_AGENT_JOM_WORKSPACE_ROOT"] = str(TARGET_WORKSPACE_ROOT)

if str(AGENT_SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(AGENT_SRC_ROOT))

import swe_agent_jom.config.settings as settings_module
import swe_agent_jom.runtime.workspace as workspace_module

# Notebook kernels keep imported modules alive across reruns, so update the
# runtime globals directly after setting the target workspace env var.
settings_module.WORKSPACE_ROOT = TARGET_WORKSPACE_ROOT.resolve()
workspace_module.WORKSPACE_ROOT = settings_module.WORKSPACE_ROOT

from swe_agent_jom.core.tool_result import ToolResult, error_result
from swe_agent_jom.memory.trajectory import Trajectory
from swe_agent_jom.tools.file_tool import list_files, read_file
from swe_agent_jom.tools.search_tool import search_code

## 配置 LLM client

这里沿用前面 prototype 的 DeepSeek/OpenAI-compatible client。运行前需要 `.env` 里有 `DEEPSEEK_API_KEY`。

In [2]:
load_dotenv(PROJECT_ROOT / ".env")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

if not DEEPSEEK_API_KEY:
    raise RuntimeError("Missing DEEPSEEK_API_KEY. Please add it to your .env file.")

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com",
)

MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash")

print("MODEL:", MODEL)

MODEL: deepseek-v4-flash


## 定义 read-only tool schemas

LLM 只能看到这三个只读工具。没有 write、patch、command runner。

In [3]:
AVAILABLE_TOOL_NAMES = ["list_files", "read_file", "search_code"]


TOOLS: list[ChatCompletionToolParam] = [
    cast(ChatCompletionToolParam, {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files and directories inside the target workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Workspace-relative path."},
                    "pattern": {"type": "string", "description": "Glob pattern, such as *.py."},
                    "max_results": {"type": "integer", "description": "Maximum entries to return."},
                },
            },
        },
    }),
    cast(ChatCompletionToolParam, {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a text file inside the target workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Workspace-relative file path."},
                    "max_chars": {"type": "integer", "description": "Maximum characters to read."},
                },
                "required": ["path"],
            },
        },
    }),
    cast(ChatCompletionToolParam, {
        "type": "function",
        "function": {
            "name": "search_code",
            "description": "Search text inside files in the target workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Text to search for."},
                    "path": {"type": "string", "description": "Workspace-relative file or directory."},
                    "pattern": {"type": "string", "description": "Glob pattern, such as *.py."},
                    "case_sensitive": {"type": "boolean"},
                    "max_results": {"type": "integer"},
                },
                "required": ["query"],
            },
        },
    }),
]

## 工具 dispatch

模型只产生 tool call；真正执行工具的是 Python。这里是安全边界：未知工具会被拒绝。

In [4]:
def parse_tool_arguments(raw_arguments: str) -> dict[str, Any]:
    try:
        parsed = json.loads(raw_arguments or "{}")
    except json.JSONDecodeError as error:
        raise ValueError(f"Invalid JSON tool arguments: {error}") from error

    if not isinstance(parsed, dict):
        raise ValueError("Tool arguments must decode to a JSON object.")

    return cast(dict[str, Any], parsed)


def dispatch_tool(tool_name: str, arguments: dict[str, Any]) -> ToolResult:
    if tool_name == "list_files":
        return list_files(
            path=cast(str, arguments.get("path", ".")),
            pattern=cast(str, arguments.get("pattern", "*")),
            max_results=int(arguments.get("max_results", 50)),
        )

    if tool_name == "read_file":
        return read_file(
            path=cast(str, arguments["path"]),
            max_chars=int(arguments.get("max_chars", 2000)),
        )

    if tool_name == "search_code":
        return search_code(
            query=cast(str, arguments["query"]),
            path=cast(str, arguments.get("path", ".")),
            pattern=cast(str, arguments.get("pattern", "*")),
            case_sensitive=bool(arguments.get("case_sensitive", False)),
            max_results=int(arguments.get("max_results", 20)),
        )

    return error_result(
        "UnknownToolError",
        f"Unknown tool: {tool_name}",
        details={"available_tools": AVAILABLE_TOOL_NAMES},
    )


def tool_result_to_message(tool_call_id: str, result: ToolResult) -> ChatCompletionMessageParam:
    return cast(
        ChatCompletionMessageParam,
        {
            "role": "tool",
            "tool_call_id": tool_call_id,
            "content": json.dumps(result, ensure_ascii=False),
        },
    )

## System prompt

这个 prompt 把模型关在只读调试任务里：能查、能读、能推理，但不能假装已经修复。

In [5]:
SYSTEM_PROMPT = """
You are a read-only SWE debugging agent.

Target workspace: fixtures/workspaces/calculator_bug.

You can use these read-only tools:
- list_files
- read_file
- search_code

Rules:
1. Use tools to inspect the target workspace before making claims.
2. Do not claim you modified files. You do not have write or patch tools.
3. Do not ask to run shell commands. Command execution is intentionally out of scope here.
4. If you identify a bug, explain the failing behavior, the relevant file, and the exact suggested code change.
5. Keep the final answer concise and practical.
""".strip()

## LLM read-only agent loop

In [6]:
def assistant_message_to_dict(message: Any) -> ChatCompletionMessageParam:
    """Convert an OpenAI SDK assistant message to a messages-list item."""
    return cast(ChatCompletionMessageParam, message.model_dump(exclude_none=True))


def run_llm_read_only_debug_agent(
    task: str,
    *,
    max_turns: int = 6,
) -> dict[str, Any]:
    trajectory = Trajectory()
    trajectory.record_user_message(task)

    messages: list[ChatCompletionMessageParam] = [
        cast(ChatCompletionMessageParam, {"role": "system", "content": SYSTEM_PROMPT}),
        cast(ChatCompletionMessageParam, {"role": "user", "content": task}),
    ]

    for _ in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message_to_dict(assistant_message))

        content = assistant_message.content or ""
        if content:
            trajectory.record_assistant_message(content)

        tool_calls = assistant_message.tool_calls or []

        if not tool_calls:
            final_answer = content
            trajectory.record_final_answer(final_answer)
            return {
                "final_answer": final_answer,
                "messages": messages,
                "trajectory": trajectory.to_dicts(),
            }

        for tool_call in tool_calls:
            function_tool_call = cast(ChatCompletionMessageToolCall, tool_call)
            tool_name = function_tool_call.function.name

            try:
                arguments = parse_tool_arguments(function_tool_call.function.arguments)
            except ValueError as error:
                result = error_result("InvalidToolArgumentsError", str(error))
                arguments = {"raw_arguments": function_tool_call.function.arguments}
            else:
                result = dispatch_tool(tool_name, arguments)

            trajectory.record_tool_call(tool_name, arguments, tool_call_id=function_tool_call.id)
            trajectory.record_tool_result(
                tool_name,
                cast(dict[str, Any], result),
                tool_call_id=function_tool_call.id,
            )

            messages.append(tool_result_to_message(function_tool_call.id, result))

    final_answer = "Stopped before final answer: max_turns reached."
    trajectory.record_final_answer(final_answer)

    return {
        "final_answer": final_answer,
        "messages": messages,
        "trajectory": trajectory.to_dicts(),
    }

## 运行 LLM read-only debug agent

运行这个 cell 会调用模型 API。这个 notebook 默认不预执行输出，避免把 API 结果和私有环境写进 diff。

In [7]:
result = run_llm_read_only_debug_agent(
    "The calculator tests are failing. Inspect the workspace and identify the bug."
)

print(result["final_answer"])

APITimeoutError: Request timed out.

## 查看 trajectory

In [8]:
print(json.dumps(result["trajectory"], ensure_ascii=False, indent=2))

NameError: name 'result' is not defined

## 下一步

如果 4.3 能稳定让 LLM 找到 bug，下一步再考虑 patch/write 工具。现在先让它学会只读地观察、推理、解释。